# AlphaZero chess: train in Colab

Short notebook that installs the `alphazero` package from this repo and runs training.
The code lives in `alphazero/`; the original exploratory notebooks are in `notebooks/archive/`.

1. **Runtime → Change runtime type → GPU** (CPU also works, just slower).
2. Run the cells top to bottom. Checkpoints can go to Google Drive so a disconnected session can resume.

In [ ]:
# @title Settings
BRANCH = "main"            # @param {type:"string"}
PRESET = "colab"           # @param ["tiny", "small", "colab"]
ITERATIONS = 20            # @param {type:"integer"}
USE_DRIVE = True           # @param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/AlphaZero_Checkpoints"  # @param {type:"string"}

In [ ]:
# Clone the repo and install the package (python-chess, numpy; torch is preinstalled in Colab).
import os
%cd /content
if not os.path.exists("alphazero-optimized"):
    !git clone --depth 1 -b {BRANCH} https://github.com/Rapheal7/alphazero-optimized.git
%cd /content/alphazero-optimized
!git pull --ff-only
%pip install -q -e .

In [ ]:
# Where to keep checkpoints.
CHECKPOINT_DIR = "/content/alphazero-optimized/checkpoints"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = DRIVE_DIR
print("Checkpoints:", CHECKPOINT_DIR)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (running on CPU)")

## Train
`--resume` picks up from the latest checkpoint in the folder, so re-running this cell after a disconnect continues where it stopped.
Run `!alphazero train --help` to see every option (simulations, network size, games per iteration, ...).

In [ ]:
!alphazero train --preset {PRESET} --iterations {ITERATIONS} --checkpoint-dir "{CHECKPOINT_DIR}" --resume

## Evaluate
Against a random mover (no setup), or against Stockfish (installed with apt below).

In [ ]:
!alphazero evaluate --checkpoint-dir "{CHECKPOINT_DIR}" --opponent random --games 4

In [ ]:
!apt-get -qq install -y stockfish > /dev/null
!alphazero evaluate --checkpoint-dir "{CHECKPOINT_DIR}" --opponent stockfish --stockfish-path /usr/games/stockfish --skill-level 0 --games 4

## Play against it
Uses the Python API so it works with Colab's input box. Enter moves as SAN (`e4`, `Nf3`) or UCI (`e2e4`).

In [ ]:
from alphazero.cli import main
main(["play", "--checkpoint-dir", CHECKPOINT_DIR, "--color", "white"])